# 02 — Kaggle inference B0, Qwen3-Coder Q4_K_M
Notebook tự chứa runtime. Bật **Internet + GPU T4 ×2**. Add Input dataset prompt từ notebook 01, model GGUF nếu có, và checkpoint B0 của phiên trước nếu muốn resume.
Không cần sửa username hoặc slug. Tự quét input theo manifest B0, model hash và tên checkpoint; nếu nhiều phiên bản xung đột sẽ báo rõ.
Giống B2: giữ model trong bộ nhớ qua các split, lấy logits gốc, fsync từng record, dừng theo ngân sách và đóng gói checkpoint kể cả khi lỗi. Không ép grammar.


> Future Kaggle runs use the shared package lock (SHA-256 `bf1f859dc786e206a3c11d655b04b528dc68f59a15a82d4db894ac5944b0ac2e`), including SciPy 1.16.3. Existing outputs in this notebook document earlier runs and are not retroactively changed.


## Tự khóa ngưỡng validation trước test
Khi đủ validation của arm/dataset, runtime tự chọn tau theo MCC (lưới 0,01–0,99), lưu thresholds.json vào thư mục checkpoint. Test chỉ được xử lý khi có ngưỡng cùng model/tokenizer/backend. Chia phiên/resume vẫn dùng checkpoint cũ; nếu đã inference test ở phiên cũ thì ngưỡng chốt sau đó không biến run cũ thành đăng ký trước. Cập nhật notebook không thay mã đang chạy trên Kaggle.


In [1]:
from pathlib import Path
import sys, os, json, subprocess, traceback, shutil, platform, importlib.metadata, zipfile, hashlib, time, re
from datetime import datetime, timezone

DIAG = Path('/kaggle/working/grace_b0_diagnostics')
DIAG.mkdir(parents=True, exist_ok=True)
EVENTS = DIAG / 'events.log'

def note(message):
    text = str(message)
    print(text, flush=True)
    with EVENTS.open('a', encoding='utf-8') as f:
        f.write(f'[{datetime.now(timezone.utc).isoformat()}] {text}\n')

def diagnostics_zip():
    archive = shutil.make_archive('/kaggle/working/grace_b0_diagnostics', 'zip', root_dir=DIAG)
    print('Tải log chẩn đoán từ Kaggle Output:', archive, flush=True)
    return archive

def run_logged(cmd, name, env=None):
    log = DIAG / f'{name}.log'
    note(f'Bắt đầu {name}: {cmd!r} | log đầy đủ: {log}')
    is_inference = name == 'inference_batch'
    with log.open('w', encoding='utf-8') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, encoding='utf-8', errors='replace', bufsize=1, env=env)
        for line in proc.stdout:
            f.write(line); f.flush()
            if re.search(r'\b(?:Found|Skip(?:ping)?) link\b', line, flags=re.IGNORECASE):
                continue
            if is_inference:
                match = re.search(r'^(\S+).*\((\d+)/(\d+)\)\s*$', line.strip())
                if match:
                    run_key, done, total = match.group(1), int(match.group(2)), int(match.group(3))
                    if done % 100 == 0 or done == total:
                        print(f'Inference {run_key}: {done}/{total}', flush=True)
                elif line.strip():
                    print(line, end='', flush=True)
            else:
                print(line, end='', flush=True)
        code = proc.wait()
    if is_inference and '--status' in cmd:
        status_path = Path(cmd[cmd.index('--status') + 1])
        if status_path.is_file():
            status = json.loads(status_path.read_text(encoding='utf-8'))
            print(f"Inference summary: {status.get('completed_records_total', 0)} records; state={status.get('state')}", flush=True)
            for run_key, progress in status.get('runs', {}).items():
                print(f"  {run_key}: {progress.get('completed', 0)}/{progress.get('count', '?')} [{progress.get('state', 'pending')}]")
    note(f'{name} kết thúc với mã {code}')
    if code:
        raise subprocess.CalledProcessError(code, cmd)

def versions(label='Installed package versions'):
    result = {}
    for name in ('llama-cpp-python', 'transformers', 'huggingface_hub', 'scikit-learn', 'numpy', 'scipy'):
        try: result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: result[name] = 'not installed'
    note(label + ': ' + json.dumps(result))
    return result

def capture_failure(shell, etype, evalue, tb, tb_offset=None):
    trace = ''.join(traceback.format_exception(etype, evalue, tb))
    try:
        with (DIAG / 'exceptions.log').open('a', encoding='utf-8') as f:
            f.write(f'\n[{datetime.now(timezone.utc).isoformat()}]\n{trace}\n')
        note(f'Lỗi: {etype.__name__}: {evalue}')
        versions()
        diagnostics_zip()
    except Exception as diagnostic_error:
        print('Không thể ghi đầy đủ chẩn đoán:', diagnostic_error, flush=True)
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    return []

get_ipython().set_custom_exc((Exception,), capture_failure)
note('UTC: ' + datetime.now(timezone.utc).isoformat())
note('Python: ' + sys.version)
note('Executable: ' + sys.executable)
note('Platform: ' + platform.platform())
for loc in ('/kaggle/working', '/kaggle/input'):
    try:
        usage = shutil.disk_usage(loc)
        note(f'Disk {loc}: total={usage.total} free={usage.free} bytes')
    except OSError as exc: note(f'Disk {loc}: {exc}')
versions('Kaggle base image package versions before pinned installation')
for name, cmd in [('gpu', ['nvidia-smi']), ('cuda_compiler', ['nvcc', '--version']), ('cmake', ['cmake', '--version']), ('pip', [sys.executable, '-m', 'pip', '--version'])]:
    try: run_logged(cmd, name)
    except (OSError, subprocess.CalledProcessError) as exc: note(f'{name}: {type(exc).__name__}: {exc}')
diagnostics_zip()


# Shared Kaggle Qwen package contract. The initial package log above describes
# the Kaggle base image; inference starts only after the pinned install cell.
RUNTIME_LOCK_BYTES = b'{\n  "backend_source": {\n    "allow_other_backend": false,\n    "name": "llama-cpp-python",\n    "record_runner_source_sha256": true,\n    "sdist_filename": "llama_cpp_python-0.3.16.tar.gz",\n    "sdist_sha256": "34ed0f9bd9431af045bb63d9324ae620ad0536653740e9bb163a2e1fcb973be6",\n    "verify_sdist_before_build": true,\n    "version": "0.3.16"\n  },\n  "build": {\n    "CMAKE_BUILD_PARALLEL_LEVEL": "2",\n    "CMAKE_CUDA_ARCHITECTURES": "75",\n    "CMAKE_LIBRARY_PATH": "/usr/local/nvidia/lib64",\n    "FORCE_CMAKE": "1",\n    "GGML_CUDA": true,\n    "binary_environment_fully_locked": false,\n    "compiled_wheel_sha256": null,\n    "exact_cmake": null,\n    "exact_cuda_toolkit": null,\n    "exact_host_compiler": null,\n    "exact_python_patch": null,\n    "policy": "Kaggle platform build is approved. Record exact Python patch, CUDA/nvcc/driver, compiler, CMake, flags, dependencies and compiled backend hash. A shared container or bit-identical binary is not required or claimed.",\n    "python_family": "3.13"\n  },\n  "constructor": {\n    "embedding": false,\n    "flash_attn": false,\n    "logits_all": true,\n    "main_gpu": 0,\n    "n_batch": 256,\n    "n_ctx": 8192,\n    "n_gpu_layers": -1,\n    "n_ubatch": 256,\n    "offload_kqv": true,\n    "seed": 4294967295,\n    "split_mode": "LLAMA_SPLIT_MODE_LAYER",\n    "tensor_split": null,\n    "use_mlock": false,\n    "use_mmap": true\n  },\n  "contract_id": "grace_qwen_q4_runtime_v1",\n  "created_on": "2026-11-07",\n  "decoding": {\n    "generation_api": null,\n    "mode": "direct_logits_and_greedy_argmax",\n    "note": "Sampling parameters are not used by eval; the default-seed sentinel does not promise fixed RNG output for future sampling paths.",\n    "sampling": false,\n    "temperature": null,\n    "top_k": null,\n    "top_p": null\n  },\n  "gpu": {\n    "allow_auxiliary_host_buffers": true,\n    "allow_silent_cpu_layer_fallback": false,\n    "device_count": 2,\n    "device_model": "NVIDIA Tesla T4",\n    "distribution": "layer_by_free_vram_at_load",\n    "fixed_ratio_guaranteed": false,\n    "other_hardware_requires_explicit_profile_review": true,\n    "profile": "kaggle_t4_x2",\n    "random_distribution": false,\n    "record_device_order_and_uuid": true,\n    "record_free_and_total_vram": true,\n    "record_layer_and_buffer_offload_report": true,\n    "required_offloaded_layers": 49\n  },\n  "integration": {\n    "existing_notebooks_auto_enforce_json": false,\n    "historical_outputs_retroactively_certified": false,\n    "historical_runs_are_not_retroactively_certified": true,\n    "package_contract_applies_to_all_registered_kaggle_arms": true,\n    "required_before_claiming_compliance": "load, validate and apply this JSON; abort on incompatible values"\n  },\n  "integrity": {\n    "algorithm": "sha256",\n    "canonicalization": "UTF-8 json.dumps(sort_keys=True,separators=(comma,colon),ensure_ascii=False); exclude integrity",\n    "sha256": "d79396de5d37ca6c3123ac05ea122d665f5f9b252aa0b32ac2b5c464b68e338f"\n  },\n  "model": {\n    "adapters_allowed": false,\n    "filename": "Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf",\n    "quantization": "Q4_K_M",\n    "repo": "ggml-org/Qwen3-Coder-30B-A3B-Instruct-GGUF",\n    "sha256": "f6050b4da22869aa3fb9d94bb3314e9fc5a027d4741c1f97761c387d6b2b2335",\n    "verify_sha256_before_load": true\n  },\n  "notebook_audit": [\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "d83c14d4f069e1987c024b27ac801840f7fa080d6fd3f75a17f2e004fdce9ba3",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "01_Author_Baseline_Unified/notebooks/02_kaggle_qwen_q4_b2.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 42,\n      "file_sha256_at_audit": "fdc0ee82e6df6903463023471e2babef6fd6a136bfc4ce14e7dd8806376ee7d6",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "02_Arm_B0_Source_Only/notebooks/02_kaggle_qwen_q4_b0.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 42,\n      "file_sha256_at_audit": "c01dac2696b9391414d0f9c5ee0f7eb1f26e1ef0f69de13b953da83f75f18746",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "03_Arm_B1_Graph_Only/notebooks/02_kaggle_qwen_q4_b1.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 42,\n      "file_sha256_at_audit": "6e501ab2f57b964e9c43a807297df4407620bcb9a02f0a3744d742341dce22fb",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "03_Arm_B1_Graph_Only/notebooks/02_kaggle_qwen_q4_b1_executed_I08.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 42,\n      "file_sha256_at_audit": "c6e7ce9c1351ab88375cd6e4d9b7d34cd5bb9f7862108fef5d12d81196cf37fb",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "04_Arm_B2_Flip/notebooks/02_kaggle_qwen_q4_b2_flip.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "b819048b5f126882dba6ff59581354937101195d8807974bc3a1f7cc65910c86",\n      "missing_explicit_package_pins": [],\n      "path": "06_Arm_B2_Rand/notebooks/02_kaggle_qwen_q4_b2_rand.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "a35b30da7d32ca935f0d5bba894ccb4fa02ba52e1212c70d6afded2ba08e3c7e",\n      "missing_explicit_package_pins": [],\n      "path": "06_Arm_B2_Rand/test_delivery_20261107/02_kaggle_qwen_q4_b2_rand_seed42_test.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "42ebe992c4e716ced593efdd089a720d3d0019663b69f5faf303c201975264d1",\n      "missing_explicit_package_pins": [],\n      "path": "06_Arm_B2_Rand/test_delivery_20261107/02_kaggle_qwen_q4_b2_rand_seed43_test.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "3cb1f4d4fc2d4e9ca839d22db2b1660f671aebbd046bdd2a0e2e94f4a8541dc7",\n      "missing_explicit_package_pins": [],\n      "path": "11_Arm_B1H2_Evidence/kaggle_b1h2_test_20261106/02_kaggle_qwen_q4_b1_h2_test_20261106.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": [\n        1.0,\n        1.0\n      ]\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "57cdf5249395aec0f0c5bc47421a9d6fb3789fd5809de7d42bc80f632d91c88f",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "11_Arm_B1H2_Evidence/notebooks/02_kaggle_qwen_q4_b1_h2_validation_20261106.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": [\n        1.0,\n        1.0\n      ]\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "dcfff73568daddd1e97934c0f2175d4b36b36339032049fffb16160a541b34bf",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b1_full_validation.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "d9ed03125e8f309e28b5f06d14be3038e3fe77277fe9ae0ac33857d777aa9759",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b1_shuf_validation.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "6a54f85e4537e66ab1a1c57615232e3b8a99850a70308c3d1bcadb0fcdb3d8db",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_labelonly_validation.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "099325d9a24e2df72e43a33f77250ea1590302409d7d5cb1b5a7a035f605364a",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_nolabel_validation.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "ccef18aeb640a0ded8d5050e54e58a6e504cd0986460288425fba7a5c7dfed28",\n      "missing_explicit_package_pins": [],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_rand.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    },\n    {\n      "configured_inference_seed": 4294967295,\n      "file_sha256_at_audit": "2e6bfb53c9fc4525a014a9d79ad862b5c3fbc40fd3bcf8e24729ab91257e7a8e",\n      "missing_explicit_package_pins": [\n        "numpy",\n        "huggingface-hub",\n        "tokenizers"\n      ],\n      "path": "Runs-management/kaggle_validation_6arms_20261104/02_kaggle_qwen_q4_b2_rep_validation.ipynb",\n      "shared_json_integrated": false,\n      "tensor_split": null\n    }\n  ],\n  "open_questions": [\n    {\n      "arms": [\n        "B1-full",\n        "B1-shuf",\n        "B2-rep",\n        "B2-nolabel",\n        "B2-labelonly"\n      ],\n      "id": "missing_run_logs",\n      "status": "awaiting executed notebooks/install logs"\n    },\n    {\n      "arms": [\n        "B3",\n        "B3-nodiff",\n        "B3-swap",\n        "B3-flip"\n      ],\n      "id": "b3_runners",\n      "status": "no dedicated current Kaggle Qwen notebook found in GRACE-summary"\n    }\n  ],\n  "package_contract_scope": "All registered Kaggle Qwen Q4 inference arms (B0, B1, B1-full, B1-H2, B1-shuf, B2, B2-rand, B2-nolabel, B2-labelonly, B2-flip, B2-rep, B3, B3-nodiff, B3-swap, B3-flip). Apply to future runs; historical outputs retain their recorded environment.",\n  "packages": {\n    "huggingface-hub": "0.36.2",\n    "llama-cpp-python": "0.3.16",\n    "numpy": "2.1.3",\n    "scikit-learn": "1.5.2",\n    "scipy": "1.16.3",\n    "tokenizers": "0.22.2",\n    "transformers": "4.57.1"\n  },\n  "prompt_contract": {\n    "different_arms_use_their_own_frozen_prompts": true,\n    "identity": [\n      "arm",\n      "prompt seed",\n      "dataset",\n      "split",\n      "ID",\n      "bundle SHA256",\n      "prompt SHA256"\n    ],\n    "prompt_selection_seeds": [\n      42,\n      43,\n      44\n    ],\n    "selection_seed_distinct_from_inference_seed": true,\n    "silent_reformatting_or_truncation": false,\n    "verify_frozen_input_and_each_prompt_hash": true\n  },\n  "provenance_required": [\n    "contract SHA256",\n    "all package versions including transitive dependencies",\n    "exact Python/OS/libc",\n    "CUDA toolkit/nvcc/driver",\n    "GPU device order/UUID/VRAM",\n    "host compiler/CMake/build flags",\n    "source archive SHA256",\n    "compiled backend/wheel SHA256",\n    "runner source SHA256",\n    "effective CPU threads",\n    "effective batch/microbatch",\n    "offload log",\n    "model/tokenizer/prompt/threshold identities"\n  ],\n  "references": {\n    "auto_split": "https://raw.githubusercontent.com/ggml-org/llama.cpp/4227c9be4268ac844921b90f31595f81236bd317/src/llama-model.cpp",\n    "constructor_defaults": "https://raw.githubusercontent.com/abetlen/llama-cpp-python/v0.3.16/llama_cpp/llama.py",\n    "package_hash": "https://pypi.org/pypi/llama-cpp-python/0.3.16/json"\n  },\n  "resolved_decisions": [\n    {\n      "decision": "Kaggle platform build with mandatory full provenance",\n      "id": "build_artifact",\n      "question": "Pin a shared container/build artifact or allow platform builds with provenance? All notebook builders leave exact CUDA/compiler/CMake/Python patch unpinned.",\n      "status": "resolved"\n    },\n    {\n      "decision": "preserve default CPU-count formulas and record effective counts",\n      "id": "cpu_threads",\n      "interim_policy": "preserve existing formulas and record effective values",\n      "question": "Keep CPU-count default formulas or lock absolute thread counts? All 16 constructors omit n_threads and n_threads_batch.",\n      "status": "resolved"\n    }\n  ],\n  "resume": {\n    "changed_contract": "do not silently append to historical results; use separate run identity or reviewed migration",\n    "require_match": [\n      "arm",\n      "prompt seed",\n      "dataset",\n      "split",\n      "ID",\n      "label",\n      "prompt SHA256",\n      "model SHA256",\n      "tokenizer revision",\n      "backend source",\n      "scoring",\n      "constructor",\n      "threshold lock when applicable"\n    ],\n    "rewrite_historical_outputs": false\n  },\n  "schema_version": 1,\n  "scope": "New GRACE diagnostic Qwen Q4 inference on Kaggle only; excludes Colab/Vast and preparation/CodeT5/legacy API prototypes",\n  "scoring": {\n    "evaluate": "llm.eval(gguf_ids)",\n    "logits": "llm.scores[llm.n_tokens - 1]",\n    "logprob": "stable full-vocabulary log_softmax of last-position logits",\n    "name": "unconstrained_first_token_logits_v1",\n    "normalization_dtype": "float64",\n    "pred_argmax": "int(s >= 0.5)",\n    "raw_token": "argmax of full-vocabulary logits",\n    "reject_non_finite": true,\n    "reset_before_each_prompt": true,\n    "retain_valid_score_if_raw_not_binary": true,\n    "s": "exp(logprob_1 - logaddexp(logprob_0, logprob_1))"\n  },\n  "status": "shared_package_contract_locked_notebook_adoption_in_progress",\n  "threads": {\n    "approval_status": "user_approved",\n    "fixed_thread_count": null,\n    "n_threads": "max(multiprocessing.cpu_count() // 2, 1)",\n    "n_threads_batch": "multiprocessing.cpu_count()",\n    "policy": "preserve_backend_default_formulas",\n    "record_effective_counts": true\n  },\n  "thresholds": {\n    "preserve_existing_probe_inheritance": true,\n    "source": "existing frozen arm-specific threshold contracts",\n    "test_refit_allowed": false\n  },\n  "timezone": "Asia/Bangkok",\n  "tokenizer": {\n    "chat_template": {\n      "add_generation_prompt": true,\n      "role": "user"\n    },\n    "gguf_tokenize": {\n      "add_bos": false,\n      "special": true\n    },\n    "repo": "Qwen/Qwen3-Coder-30B-A3B-Instruct",\n    "require_hf_gguf_token_id_equality": true,\n    "require_single_token_binary_labels": true,\n    "revision": "b2cff646eb4bb1d68355c01b18ae02e7cf42d120"\n  },\n  "user_decisions": {\n    "build": "approved: Kaggle platform build with mandatory complete provenance; no shared container required",\n    "cpu_threads": "approved: preserve default CPU-count formulas used by all 16 notebooks",\n    "gpu_policy": "layer + auto free VRAM, approved",\n    "inference_seed": 4294967295,\n    "packages": "exact locked versions approved, including scipy==1.16.3 on 2026-11-07"\n  }\n}\n'
RUNTIME_LOCK_SHA256 = 'bf1f859dc786e206a3c11d655b04b528dc68f59a15a82d4db894ac5944b0ac2e'
RUNTIME_LOCK_PATH = Path('/kaggle/working/inference_runtime_lock.json')
if hashlib.sha256(RUNTIME_LOCK_BYTES).hexdigest() != RUNTIME_LOCK_SHA256:
    raise RuntimeError('Embedded runtime lock bytes/hash mismatch')
if RUNTIME_LOCK_PATH.exists() and RUNTIME_LOCK_PATH.read_bytes() != RUNTIME_LOCK_BYTES:
    raise RuntimeError('A different runtime lock is already present; restart Kaggle session')
RUNTIME_LOCK_PATH.write_bytes(RUNTIME_LOCK_BYTES)
RUNTIME_LOCK = json.loads(RUNTIME_LOCK_BYTES)
_unsigned_runtime_lock = {k:v for k,v in RUNTIME_LOCK.items() if k != 'integrity'}
_actual_integrity = hashlib.sha256(json.dumps(_unsigned_runtime_lock,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
if _actual_integrity != RUNTIME_LOCK['integrity']['sha256']:
    raise RuntimeError('Embedded runtime lock internal integrity mismatch')


UTC: 2026-11-01T04:00:59.000259+00:00
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Executable: /usr/bin/python3
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
Disk /kaggle/working: total=20957446144 free=20940464128 bytes
Disk /kaggle/input: total=20957446144 free=20940464128 bytes
Installed package versions: {"llama-cpp-python": "not installed", "transformers": "5.16.1", "huggingface_hub": "1.29.0", "scikit-learn": "1.6.1", "numpy": "2.1.3", "scipy": "1.16.3"}
Bắt đầu gpu: ['nvidia-smi'] | log đầy đủ: /kaggle/working/grace_b0_diagnostics/gpu.log
Sun Nov  1 04:00:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:U

'/kaggle/working/grace_b0_diagnostics.zip'

## Cấu hình


In [2]:
INPUT_ROOT=Path('/kaggle/input')
OUT_ROOT=Path('/kaggle/working/grace_b0_runs')
EXPECTED_RUNS=[('devign','validation'),('reveal','validation'),('devign','test'),('reveal','test')]
# Có thể chỉ chạy validation trước; bỏ hai tuple test ở trên.
MAX_RUNTIME_HOURS=8
CHECKPOINT_RESERVE_SECONDS=300
GPU_LAYERS=-1
LIMIT=None  # None = toàn split; thay đổi LIMIT phải dùng thư mục OUT_ROOT mới.
RESUME=True
MODEL_OVERRIDE=None  # Tùy chọn: file GGUF gắn Input; không cần nhập username.
ALLOW_HUB_DOWNLOAD=True
assert MAX_RUNTIME_HOURS*3600>CHECKPOINT_RESERVE_SECONDS>0


## Cài backend CUDA như baseline B2


In [3]:
# Cài từng nhóm để nhận diện chính xác bước thất bại; giữ nguyên phiên bản baseline.
# T4 dùng SM75; Kaggle đặt libcuda.so trong /usr/local/nvidia/lib64 nên thêm thư mục này cho CMake.
os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_LIBRARY_PATH=/usr/local/nvidia/lib64'
os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] = '2'
os.environ['FORCE_CMAKE'] = '1'
note('Build config: CMAKE_ARGS=' + os.environ['CMAKE_ARGS'] + '; CMAKE_BUILD_PARALLEL_LEVEL=' + os.environ['CMAKE_BUILD_PARALLEL_LEVEL'] + '; FORCE_CMAKE=' + os.environ['FORCE_CMAKE'])
# Install every package pinned by the shared package contract.
import urllib.request
RUNTIME_PACKAGES = RUNTIME_LOCK['packages']
packages = [f"{name}=={version}" for name, version in RUNTIME_PACKAGES.items() if name != 'llama-cpp-python']
packages += ['typing-extensions>=4.5.0', 'diskcache>=5.6.1', 'jinja2>=2.11.3']
run_logged([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', *packages], 'install_dependencies')
source = RUNTIME_LOCK['backend_source']
with urllib.request.urlopen(f"https://pypi.org/pypi/llama-cpp-python/{source['version']}/json") as response:
    pypi_metadata = json.load(response)
artifact = next(item for item in pypi_metadata['urls'] if item['filename'] == source['sdist_filename'])
if artifact['digests']['sha256'] != source['sdist_sha256']:
    raise RuntimeError('llama-cpp-python source hash does not match the runtime contract')
archive_path = Path('/kaggle/working') / source['sdist_filename']
urllib.request.urlretrieve(artifact['url'], archive_path)
if hashlib.sha256(archive_path.read_bytes()).hexdigest() != source['sdist_sha256']:
    raise RuntimeError('Downloaded llama-cpp-python source archive failed SHA-256 verification')
run_logged([sys.executable, '-m', 'pip', 'install', '-v', '--no-cache-dir', '--no-deps', '--force-reinstall', str(archive_path)], 'install_llama_cuda')
import importlib.metadata as _metadata
actual_locked = {name: _metadata.version(name) for name in RUNTIME_PACKAGES}
if actual_locked != RUNTIME_PACKAGES:
    raise RuntimeError(f'Installed package versions differ from shared lock: {actual_locked}')
(DIAG / 'inference_runtime_lock.json').write_bytes(RUNTIME_LOCK_BYTES)
(DIAG / 'locked_package_versions.json').write_text(json.dumps(actual_locked, indent=2), encoding='utf-8')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze', '--all'], check=True, capture_output=True, text=True)
(DIAG / 'pip_freeze_all.txt').write_text(freeze.stdout, encoding='utf-8')
note('Locked package versions: ' + json.dumps(actual_locked, sort_keys=True))
versions()
diagnostics_zip()

Build config: CMAKE_ARGS=-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_LIBRARY_PATH=/usr/local/nvidia/lib64; CMAKE_BUILD_PARALLEL_LEVEL=2; FORCE_CMAKE=1
Bắt đầu install_dependencies: ['/usr/bin/python3', '-m', 'pip', 'install', '--no-cache-dir', 'transformers==4.57.1', 'huggingface_hub', 'scikit-learn==1.5.2'] | log đầy đủ: /kaggle/working/grace_b0_diagnostics/install_dependencies.log
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 23.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 79.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 141.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.4 MB/s  0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingfac

'/kaggle/working/grace_b0_diagnostics.zip'

In [4]:
import llama_cpp
versions()
note('llama-cpp-python GPU offload: ' + str(llama_cpp.llama_supports_gpu_offload()))
assert llama_cpp.llama_supports_gpu_offload(), 'llama-cpp-python chưa có CUDA; xem install_llama_cuda.log'
diagnostics_zip()


Installed package versions: {"llama-cpp-python": "0.3.16", "transformers": "4.57.1", "huggingface_hub": "0.36.2", "scikit-learn": "1.5.2", "numpy": "2.1.3", "scipy": "1.16.3"}


ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 2 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
  Device 1: Tesla T4, compute capability 7.5, VMM: yes


llama-cpp-python GPU offload: True
Tải log chẩn đoán từ Kaggle Output: /kaggle/working/grace_b0_diagnostics.zip


'/kaggle/working/grace_b0_diagnostics.zip'

## Runtime B0 (đã nhúng, không cần repo local)


In [ ]:
%%writefile /kaggle/working/validation_lock.py
"""CPU-only, validation-only threshold locking. Portable into Kaggle notebooks."""
import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path

RULES={'version':'validation_mcc_grid_v1','grid':[i/100 for i in range(1,100)],
       'objective':'MCC on finite valid validation scores only','comparison':'score >= threshold',
       'tie_break':'within 1e-9: closest to 0.5, then lower threshold',
       'degenerate':'if every MCC is undefined, lock 0.5 with MCC=null; report explicitly',
       'coverage':'require every validation ID; invalid rows retained in coverage but excluded from fitting',
       'test_usage':'never read test records or metrics to choose threshold'}
def digest_json(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
RULES_SHA256=digest_json(RULES)

def file_sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    with tmp.open('w',encoding='utf-8',newline='\n') as f:
        json.dump(value,f,ensure_ascii=False,indent=2,allow_nan=False);f.write('\n');f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)

def mcc(y,p):
    tp=sum(a==1 and b==1 for a,b in zip(y,p));tn=sum(a==0 and b==0 for a,b in zip(y,p))
    fp=sum(a==0 and b==1 for a,b in zip(y,p));fn=sum(a==1 and b==0 for a,b in zip(y,p))
    den=math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return (tp*tn-fp*fn)/den if den else None

def pick_threshold(rows):
    valid=[]
    for r in rows:
        value=r.get('s')
        if r.get('valid') and isinstance(value,(int,float)) and math.isfinite(value):
            if not 0<=value<=1:raise ValueError('Score outside [0,1]')
            valid.append(r)
    if not valid:raise ValueError('No finite valid validation scores; cannot lock threshold')
    if {r['label'] for r in valid}!={0,1}:raise ValueError('Validation fitting needs both classes among valid scores')
    y=[r['label'] for r in valid];scores=[r['s'] for r in valid]
    best_t=.5;best_m=None;grid=[]
    for t in RULES['grid']:
        value=mcc(y,[int(v>=t) for v in scores]);grid.append({'threshold':t,'mcc':value})
        if value is None:continue
        if best_m is None or value>best_m+1e-9 or (abs(value-best_m)<=1e-9 and (abs(t-.5),t)<(abs(best_t-.5),best_t)):
            best_t,best_m=t,value
    return {'tau':best_t,'validation_mcc':best_m,'n_valid':len(valid),'n_invalid':len(rows)-len(valid),
            'score_coverage':len(valid)/len(rows),'degenerate':best_m is None,'grid_results':grid}

def inference_identity(manifest):
    return {'model_sha256':manifest['model_sha256'],'tokenizer':manifest['tokenizer'],
            'tokenizer_revision':manifest['tokenizer_revision'],'quantization':manifest['quantization'],
            'backend':'llama-cpp-python','backend_version':manifest['llama_cpp_python'],
            'scoring':manifest['scoring'],'seed_recorded':42,'context_tokens':manifest['context_tokens']}

def validate_validation(expected,records,arm,dataset,model_sha256):
    exp={str(r['id']):r for r in expected}
    if len(exp)!=len(expected):raise ValueError('Duplicate expected validation IDs')
    seen={}
    for r in records:
        uid=str(r['id'])
        if r.get('split')!='validation' or r.get('dataset')!=dataset or r.get('arm')!=arm:
            raise ValueError('Only the requested arm/dataset validation records may be fitted')
        if uid in seen:raise ValueError('Duplicate validation records')
        e=exp.get(uid)
        if e is None or r.get('label')!=e['label'] or r.get('prompt_sha256')!=e['prompt_sha256']:
            raise ValueError('Validation ID/label/prompt identity mismatch')
        if r.get('model_sha256')!=model_sha256 or r.get('seed')!=42:
            raise ValueError('Validation model/seed mismatch')
        seen[uid]=r
    return exp,seen

def make_lock(expected,records,arm,dataset,run_manifest,validation_prompt_set_sha256,source_evidence=None):
    exp,seen=validate_validation(expected,records,arm,dataset,run_manifest['model_sha256'])
    if set(exp)!=set(seen):raise ValueError(f'Incomplete validation: {len(seen)}/{len(exp)}; do not fit on partial data')
    ordered=[seen[uid] for uid in sorted(seen)]
    result=pick_threshold(ordered)
    evidence=[{k:r.get(k) for k in ('id','label','prompt_sha256','model_sha256','seed','s','valid')} for r in ordered]
    # Normalize invalid scores for strict JSON; their raw validity remains recorded.
    for r in evidence:
        if r['s'] is not None and not math.isfinite(r['s']):r['s']=None
    return {'arm':arm,'dataset':dataset,'split_used':'validation','locked_at_utc':datetime.now(timezone.utc).isoformat(),
            'rules_sha256':RULES_SHA256,'identity':inference_identity(run_manifest),
            'validation_prompt_set_sha256':validation_prompt_set_sha256,'n_expected':len(exp),
            'validation_evidence_sha256':digest_json(evidence),'source_evidence':source_evidence or [],
            'temporal_claim':'locked at actual time; does not certify that external test inference had not started',**result}

def persist_lock(path,entry):
    path=Path(path)
    data=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'schema':'thresholds_v1','rules':RULES,'rules_sha256':RULES_SHA256,'entries':{}}
    if data.get('rules_sha256')!=RULES_SHA256:raise ValueError('Threshold rules differ; create a new version')
    key=f"{entry['arm']}/{entry['dataset']}";old=data['entries'].get(key)
    if old:
        for k in ('rules_sha256','identity','validation_prompt_set_sha256','validation_evidence_sha256','tau'):
            if old.get(k)!=entry.get(k):raise ValueError(f'Locked threshold is immutable; conflicting {k} for {key}')
        return old
    data['entries'][key]=entry;write_json(path,data);return entry

def require_test_lock(path,arm,dataset,run_manifest,validation_prompt_set_sha256=None):
    path=Path(path)
    if not path.exists():raise ValueError('Test blocked: thresholds.json missing; complete validation first')
    data=json.loads(path.read_text(encoding='utf-8'));entry=data.get('entries',{}).get(f'{arm}/{dataset}')
    if not entry:raise ValueError(f'Test blocked: no locked threshold for {arm}/{dataset}')
    if data.get('rules_sha256')!=RULES_SHA256 or entry.get('rules_sha256')!=RULES_SHA256:
        raise ValueError('Test blocked: different threshold selection rules')
    if entry.get('identity')!=inference_identity(run_manifest):raise ValueError('Test blocked: model/tokenizer/backend differs from validation')
    if validation_prompt_set_sha256 and entry.get('validation_prompt_set_sha256')!=validation_prompt_set_sha256:
        raise ValueError('Test blocked: validation prompt set changed')
    return entry

def evaluate_at_locked_threshold(records,entry):
    valid=[r for r in records if r.get('valid') and isinstance(r.get('s'),(int,float)) and math.isfinite(r['s'])]
    y=[r['label'] for r in valid];pred=[int(r['s']>=entry['tau']) for r in valid]
    tp=sum(a==1 and b==1 for a,b in zip(y,pred));tn=sum(a==0 and b==0 for a,b in zip(y,pred))
    fp=sum(a==0 and b==1 for a,b in zip(y,pred));fn=sum(a==1 and b==0 for a,b in zip(y,pred))
    return {'tau':entry['tau'],'n':len(valid),'score_coverage':len(valid)/len(records) if records else 0,
            'mcc':mcc(y,pred),'accuracy':(tp+tn)/len(y) if y else None,
            'confusion_matrix':[[tn,fp],[fn,tp]],'threshold_locked_at_utc':entry['locked_at_utc'],
            'threshold_validation_evidence_sha256':entry['validation_evidence_sha256']}


In [5]:
%%writefile /kaggle/working/kaggle_b0_runtime.py
import argparse
import hashlib
import json
import math
import os
import platform
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

# Shared validation-only analysis; helper is embedded by the Kaggle notebook.
import sys
if str(Path(__file__).resolve().parent.parent / 'Runs-management') not in sys.path:
    sys.path.insert(0,str(Path(__file__).resolve().parent.parent / 'Runs-management'))
from validation_lock import make_lock,persist_lock,require_test_lock,evaluate_at_locked_threshold,read_jsonl,file_sha256


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_json_atomic(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    os.replace(tmp, path)


def binary_scores(logits, label_ids):
    import numpy as np
    values=np.asarray(logits,dtype=np.float64).reshape(-1)
    if len(label_ids)!=2 or label_ids[0]==label_ids[1] or min(label_ids)<0 or max(label_ids)>=len(values):
        raise ValueError('Invalid binary token IDs')
    if not np.all(np.isfinite(values)):
        raise ValueError('Non-finite logits')
    normalizer=float(np.max(values)+np.log(np.exp(values-np.max(values)).sum()))
    lp0=float(values[label_ids[0]]-normalizer); lp1=float(values[label_ids[1]]-normalizer)
    s=float(np.exp(lp1-np.logaddexp(lp0,lp1)))
    top=np.argsort(values)[-5:][::-1]
    return {'logprob_0':lp0,'logprob_1':lp1,'p0':math.exp(lp0),'p1':math.exp(lp1),'s':s,
            'pred_argmax':int(s>=0.5),'greedy_token_id':int(np.argmax(values)),
            'top5':[{'token_id':int(i),'logprob':float(values[i]-normalizer)} for i in top]}


def load_bundle(bundle):
    with zipfile.ZipFile(bundle) as z:
        manifest = json.loads(z.read('bundle_manifest.json'))
        prompt_bytes = z.read('prompts.jsonl')
    if hashlib.sha256(prompt_bytes).hexdigest() != manifest['prompts_sha256']:
        raise ValueError(f'Prompt bundle hash mismatch: {bundle}')
    if manifest.get('arm') != 'B0' or manifest.get('prompt_variant') != 'b0_b2_empty_blocks_v1':
        raise ValueError('Expected B0 empty-B2-scaffolding bundle')
    rows = [json.loads(line) for line in prompt_bytes.decode('utf-8').splitlines() if line.strip()]
    if len(rows) != manifest['count'] or len({str(r['id']) for r in rows}) != len(rows):
        raise ValueError(f'Prompt count or unique ID check failed: {bundle}')
    for row in rows:
        if row.get('demo_ids') or row.get('arm') != 'B0' or row.get('prompt_variant') != manifest['prompt_variant']:
            raise ValueError('Bundle contains non-B0 payload/identity')
        if row.get('label') not in (0,1) or hashlib.sha256(row['prompt'].encode()).hexdigest() != row['prompt_sha256']:
            raise ValueError('Invalid prompt hash/label')
    return manifest, rows


def evaluate_output(output):
    from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                                 confusion_matrix, f1_score, matthews_corrcoef, precision_score,
                                 recall_score, roc_auc_score)
    records = [json.loads(line) for line in output.read_text(encoding='utf-8').splitlines() if line.strip()]
    actual = [r for r in records if r['response_valid']]
    truth = [r['label'] for r in actual]
    predicted = [int(r['raw']) for r in actual]
    metrics = {'n_total': len(records), 'n_score_valid': sum(bool(r['valid']) for r in records),
               'n_response_valid': len(actual),
               'response_coverage': len(actual) / len(records) if records else None,
               'accuracy': None, 'balanced_accuracy': None, 'precision': None, 'recall': None,
               'f1': None, 'mcc': None, 'confusion_matrix': None, 'auroc': None, 'auprc': None}
    if actual:
        metrics.update(accuracy=accuracy_score(truth, predicted),
                       balanced_accuracy=balanced_accuracy_score(truth, predicted),
                       precision=precision_score(truth, predicted, zero_division=0),
                       recall=recall_score(truth, predicted, zero_division=0),
                       f1=f1_score(truth, predicted, zero_division=0),
                       mcc=matthews_corrcoef(truth, predicted),
                       confusion_matrix=confusion_matrix(truth, predicted, labels=[0, 1]).tolist())
    score_rows = [r for r in records if r['valid']]
    if score_rows and len({r['label'] for r in score_rows}) == 2:
        labels = [r['label'] for r in score_rows]
        scores = [r['s'] for r in score_rows]
        metrics.update(auroc=roc_auc_score(labels, scores),
                       auprc=average_precision_score(labels, scores))
    if score_rows:
        y=[r['label'] for r in score_rows]; pred=[r['pred_argmax'] for r in score_rows]
        metrics['score_threshold_0_5']={'n':len(y),'mcc':float(matthews_corrcoef(y,pred)),
            'f1':float(f1_score(y,pred,zero_division=0)), 'accuracy':float(accuracy_score(y,pred))}
    return metrics


def run_batch(runs, model_path, model_revision, gpu_layers, limit, deadline_epoch,
              checkpoint_reserve_seconds, status_path, thresholds_path=None):
    from llama_cpp import Llama, __version__ as llama_version
    from transformers import AutoTokenizer
    if llama_version != '0.3.16': raise ValueError('Pinned llama-cpp-python 0.3.16 required')

    thresholds_path = Path(thresholds_path) if thresholds_path else Path(status_path).parent / "thresholds.json"
    started = time.time()
    stop_at = deadline_epoch - checkpoint_reserve_seconds
    status_path = Path(status_path)
    status_path.parent.mkdir(parents=True, exist_ok=True)
    status = {'state': 'starting', 'timed_out': False,
              'started_at': datetime.fromtimestamp(started, timezone.utc).isoformat(),
              'deadline_at': datetime.fromtimestamp(deadline_epoch, timezone.utc).isoformat(),
              'stop_new_records_at': datetime.fromtimestamp(stop_at, timezone.utc).isoformat(),
              'max_runtime_hours': (deadline_epoch - started) / 3600,
              'checkpoint_reserve_seconds': checkpoint_reserve_seconds,
              'runs': {}, 'completed_records_total': 0}
    for dataset, split, bundle, out in runs:
        key = f'{dataset}/{split}'
        status['runs'][key] = {'state': 'pending', 'bundle': bundle, 'output': out}
    write_json_atomic(status_path, status)

    def stop_for_deadline(current=None):
        status['state'] = 'timed_out'
        status['timed_out'] = True
        status['timed_out_at'] = datetime.now(timezone.utc).isoformat()
        status['stopped_after_run'] = current
        write_json_atomic(status_path, status)

    if time.time() >= stop_at:
        stop_for_deadline(None)
        return

    model_path = Path(model_path)
    model_digest = sha256(model_path)
    if model_revision == 'local-input':
        model_revision = f'local-sha256:{model_digest}'
    first_manifest, _ = load_bundle(runs[0][2])
    if model_digest != first_manifest['expected_model_sha256']:
        raise ValueError('GGUF SHA256 differs from frozen research model')
    for dataset, split, bundle, out in runs:
        current_manifest, _ = load_bundle(bundle)
        if current_manifest['expected_model_sha256'] != model_digest:
            raise ValueError('Model identity differs across bundles')
        if current_manifest['freeze_manifest_sha256'] != first_manifest['freeze_manifest_sha256']:
            raise ValueError('Frozen data snapshot differs across bundles')
        if (current_manifest['tokenizer'], current_manifest['tokenizer_revision'],
            current_manifest['prompt_budget_tokens']) != (first_manifest['tokenizer'],
            first_manifest['tokenizer_revision'], first_manifest['prompt_budget_tokens']):
            raise ValueError('All four bundles must use the same tokenizer revision and context budget')
    tokenizer = AutoTokenizer.from_pretrained(first_manifest['tokenizer'],
                                                revision=first_manifest['tokenizer_revision'])
    llm = Llama(model_path=str(model_path), n_ctx=first_manifest['prompt_budget_tokens'], n_batch=256,
                n_gpu_layers=gpu_layers, logits_all=True, seed=42, verbose=True)
    if time.time() >= stop_at:
        stop_for_deadline(None)
        return
    label_ids = []
    for label in ('0', '1'):
        hf_ids = tokenizer.encode(label, add_special_tokens=False)
        gguf_ids = llm.tokenize(label.encode(), add_bos=False, special=False)
        if len(hf_ids) != 1 or hf_ids != gguf_ids:
            raise ValueError(f'Label token mismatch for {label}: HF={hf_ids}, GGUF={gguf_ids}')
        label_ids.append(hf_ids[0])
    if label_ids[0] == label_ids[1]:
        raise ValueError('Label token IDs must differ')

    status['state'] = 'running'
    write_json_atomic(status_path, status)
    for dataset, split, bundle, out_text in runs:
        key = f'{dataset}/{split}'
        out = Path(out_text)
        out.mkdir(parents=True, exist_ok=True)
        manifest, rows = load_bundle(bundle)
        if (manifest['dataset'], manifest['split']) != (dataset, split):
            raise ValueError(f'Bundle identity mismatch for {key}: {manifest["dataset"]}/{manifest["split"]}')
        if limit is not None:
            rows = rows[:limit]
        identity = {'bundle_sha256': sha256(bundle),
                    'prompt_set_sha256': manifest['prompts_sha256'], 'model_sha256': model_digest,
                    'model_repo': manifest['model_repo'], 'quantization': manifest['quantization'],
                    'model_file': model_path.name, 'llama_cpp_python': llama_version,
                    'model_repo_revision': model_revision, 'tokenizer': manifest['tokenizer'],
                    'tokenizer_revision': manifest['tokenizer_revision'], 'gpu_layers': gpu_layers,
                    'context_tokens': manifest['prompt_budget_tokens'],
                    'scoring': 'unconstrained_first_token_logits_v1', 'limit': limit, 'arm': 'B0', 'prompt_variant': 'b0_b2_empty_blocks_v1',
                    'python': platform.python_version(), 'dataset': dataset, 'split': split}
        locked_threshold = None
        if split == 'test':
            val_hash = None
            for vd,vs,vb,vo in runs:
                if vd == dataset and vs == 'validation':
                    val_hash = load_bundle(vb)[0]['prompts_sha256']
                    break
            locked_threshold = require_test_lock(thresholds_path,'B0',dataset,identity,val_hash)
            print(f'Test threshold locked: {dataset} tau={locked_threshold["tau"]}',flush=True)
        identity_file = out / 'run_manifest.json'
        previous_identity = (json.loads(identity_file.read_text(encoding='utf-8'))
                             if identity_file.exists() else None)
        if previous_identity is not None:
            # Ignore only archive-byte hashes: ZIP timestamps/compression metadata may change.
            # Prompt identity is verified below against every checkpoint row's prompt_sha256.
            ignored = {'bundle_sha256', 'prompt_set_sha256'}
            previous_stable = {k: v for k, v in previous_identity.items() if k not in ignored}
            current_stable = {k: v for k, v in identity.items() if k not in ignored}
            if previous_stable != current_stable:
                raise ValueError(f'Existing output has different model/run settings for {key}; use a new OUT_ROOT')
            if ('prompt_set_sha256' in previous_identity and
                    previous_identity['prompt_set_sha256'] != identity['prompt_set_sha256']):
                raise ValueError(f'Prompt bundle content changed for {key}; refusing to resume')
        output = out / 'records.jsonl'
        done = {}
        if output.exists():
            for line in output.read_text(encoding='utf-8').splitlines():
                if line.strip():
                    row = json.loads(line)
                    row_id = str(row['id'])
                    if row_id in done:
                        raise ValueError(f'Duplicate checkpoint ID in {key}: {row_id}')
                    done[row_id] = row
        expected_ids = {str(item['id']) for item in rows}
        unexpected_ids = set(done) - expected_ids
        if unexpected_ids:
            raise ValueError(f'Checkpoint contains IDs absent from current bundle {key}: {sorted(unexpected_ids)[:10]}')
        for item in rows:
            row_id = str(item['id'])
            if row_id in done:
                if done[row_id].get('prompt_sha256') != item['prompt_sha256']:
                    raise ValueError(f'Checkpoint prompt mismatch: {key}/{row_id}')
                if done[row_id].get('label') != item['label'] or done[row_id].get('arm') != 'B0':
                    raise ValueError('Checkpoint label/arm mismatch')
                if done[row_id].get('model_sha256') != model_digest:
                    raise ValueError(f'Checkpoint model hash mismatch: {key}/{row_id}')
        # Persist the refreshed archive hash only after checkpoint IDs and prompt hashes pass.
        identity_file.write_text(json.dumps(identity, indent=2), encoding='utf-8')
        progress = status['runs'][key]
        progress.update(state='running', count=len(rows), completed=len(done),
                        records_path=str(output), bundle_sha256=identity['bundle_sha256'])
        status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
        write_json_atomic(status_path, status)
        if len(done) > len(rows):
            raise ValueError(f'Checkpoint has more rows than this bundle: {key}')
        print(f'{key}: verified checkpoint IDs={len(done)}; skipping these; remaining={len(rows) - len(done)}', flush=True)

        with output.open('a', encoding='utf-8', newline='\n') as f:
            for item in rows:
                row_id = str(item['id'])
                if row_id in done:
                    if done[row_id]['prompt_sha256'] != item['prompt_sha256']:
                        raise ValueError(f'Checkpoint prompt mismatch: {key}/{row_id}')
                    continue
                if time.time() >= stop_at:
                    progress.update(state='checkpointed', completed=len(done))
                    status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
                    stop_for_deadline(key)
                    return
                prompt = item['prompt']
                if hashlib.sha256(prompt.encode()).hexdigest() != item['prompt_sha256']:
                    raise ValueError(f'Prompt hash mismatch: {key}/{row_id}')
                messages = [{'role': 'user', 'content': prompt}]
                chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                hf_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
                gguf_ids = llm.tokenize(chat.encode('utf-8'), add_bos=False, special=True)
                if list(hf_ids) != gguf_ids:
                    raise ValueError(f'Tokenizer mismatch at {key}/{row_id}; abort rather than alter prompt')
                if len(gguf_ids) + 64 > manifest['prompt_budget_tokens']:
                    raise ValueError(f'Token budget exceeded at {key}/{row_id}')
                start = time.monotonic()
                llm.reset()
                llm.eval(gguf_ids)
                scores = binary_scores(llm.scores[llm.n_tokens - 1], label_ids)
                raw = llm.detokenize([scores['greedy_token_id']]).decode('utf-8', errors='replace').strip()
                row = {k: item[k] for k in ('id', 'dataset', 'split', 'label', 'prompt_sha256', 'demo_ids', 'tokens', 'graph_missing')}
                row.update(scores, arm='B0', prompt_variant='b0_b2_empty_blocks_v1', raw=raw, response_valid=raw in ('0', '1'), valid=True,
                           token_ids={'0': label_ids[0], '1': label_ids[1]}, model_sha256=model_digest,
                           backend='llama-cpp-python', quantization='Q4_K_M', model_repo=manifest['model_repo'],
                           model_repo_revision=model_revision, seed=42, timestamp=datetime.now(timezone.utc).isoformat(),
                           logprob_distribution='unconstrained_first_token', elapsed_s=time.monotonic()-start)
                f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + '\n')
                f.flush()
                os.fsync(f.fileno())
                done[row_id] = row
                progress['completed'] = len(done)
                status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
                status['last_checkpoint_at'] = datetime.now(timezone.utc).isoformat()
                write_json_atomic(status_path, status)
                print(f'{key} {row_id}: s={scores["s"]:.4f}, raw={raw!r} ({len(done)}/{len(rows)})', flush=True)
                if time.time() >= stop_at:
                    progress['state'] = 'checkpointed'
                    stop_for_deadline(key)
                    return

        if split == 'validation' and limit is None:
            locked_threshold = persist_lock(thresholds_path,make_lock(rows,list(done.values()),'B0',dataset,identity,
                manifest['prompts_sha256'],[{'path':str(output),'records_sha256':file_sha256(output)}]))
            write_json_atomic(out/'threshold_lock.json',locked_threshold)
            print(f'Validation threshold auto-locked: {dataset} tau={locked_threshold["tau"]}',flush=True)
        metrics = evaluate_output(output)
        if locked_threshold is not None:
            metrics['score_threshold_validation_locked'] = evaluate_at_locked_threshold(list(done.values()),locked_threshold)
        (out / 'evaluation.json').write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding='utf-8')
        progress.update(state='complete', completed=len(rows), evaluation=str(out / 'evaluation.json'))
        status['completed_records_total'] = sum(x.get('completed', 0) for x in status['runs'].values())
        write_json_atomic(status_path, status)

    status['state'] = 'complete'
    status['timed_out'] = False
    status['completed_at'] = datetime.now(timezone.utc).isoformat()
    write_json_atomic(status_path, status)



def discover_bundles(input_root,work_root,expected_runs):
    """Find standalone bundle ZIPs, unpacked bundles, or a ZIP wrapping bundle ZIPs."""
    found={}; work_root=Path(work_root); work_root.mkdir(parents=True,exist_ok=True)
    def accept(path):
        try:
            with zipfile.ZipFile(path) as z:
                if 'bundle_manifest.json' not in z.namelist(): return
                m=json.loads(z.read('bundle_manifest.json'))
        except (zipfile.BadZipFile,KeyError,json.JSONDecodeError): return
        if m.get('arm')!='B0' or m.get('prompt_variant')!='b0_b2_empty_blocks_v1': return
        key=(m['dataset'],m['split'])
        if key not in expected_runs: return
        identity=(m['prompts_sha256'],m['freeze_manifest_sha256'])
        if key in found and found[key][0]!=identity: raise ValueError(f'Conflicting B0 bundles for {key}; detach old inputs')
        found[key]=(identity,Path(path))
    for path in sorted(Path(input_root).rglob('*.zip')):
        accept(path)
        try:
            with zipfile.ZipFile(path) as z:
                nested=[n for n in z.namelist() if n.endswith('_b0_kaggle.zip')]
                for name in nested:
                    # Never trust an archive member path: extract bytes to a chosen basename.
                    dest=work_root/(hashlib.sha256(str(path).encode()).hexdigest()[:12]+'_'+Path(name).name)
                    with z.open(name) as src,dest.open('wb') as dst:
                        import shutil
                        shutil.copyfileobj(src,dst)
                    accept(dest)
        except zipfile.BadZipFile: continue
    for mp in sorted(Path(input_root).rglob('bundle_manifest.json')):
        m=json.loads(mp.read_text(encoding='utf-8'))
        if m.get('arm')!='B0' or m.get('prompt_variant')!='b0_b2_empty_blocks_v1': continue
        dest=work_root/(hashlib.sha256(str(mp).encode()).hexdigest()[:12]+'_unpacked_b0.zip')
        with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED) as z:
            z.write(mp,'bundle_manifest.json'); z.write(mp.parent/'prompts.jsonl','prompts.jsonl')
        accept(dest)
    missing=set(expected_runs)-set(found)
    if missing: raise FileNotFoundError(f'Missing B0 bundles {sorted(missing)}. Add Input: grace_b0_inputs.zip or the generated folder, under any account.')
    return {key:found[key][1] for key in expected_runs}

def discover_model(input_root,expected_sha256,override=None):
    paths=[Path(override)] if override else sorted(Path(input_root).rglob('*Q4_K_M.gguf'))
    for path in paths:
        if path.is_file() and sha256(path)==expected_sha256: return path
    if paths: raise ValueError('Attached Q4_K_M files differ from expected model SHA256; choose correct model')
    return None

def restore_checkpoint(input_root,out_root):
    out_root=Path(out_root)
    if any(out_root.rglob('records.jsonl')):
        print('Resume existing working output:',out_root); return
    candidates=sorted(Path(input_root).rglob('grace_b0_checkpoint.zip'))
    if not candidates:
        print('No B0 checkpoint attached: start new run'); return
    hashes={sha256(p) for p in candidates}
    if len(hashes)!=1: raise ValueError('Conflicting B0 checkpoints; attach only one version')
    out_root.mkdir(parents=True,exist_ok=True); root=out_root.resolve()
    with zipfile.ZipFile(candidates[0]) as z:
        for info in z.infolist():
            destination=(root/info.filename).resolve()
            if not destination.is_relative_to(root) or ((info.external_attr>>16)&0o170000)==0o120000:
                raise ValueError('Unsafe checkpoint member')
        z.extractall(root)
    print('Restored checkpoint:',candidates[0])


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--thresholds', help='Defaults to thresholds.json beside checkpoint_status.json')
    ap.add_argument('--model', required=True)
    ap.add_argument('--model-revision', required=True)
    ap.add_argument('--gpu-layers', type=int, default=-1)
    ap.add_argument('--limit', type=int)
    ap.add_argument('--deadline-epoch', type=float, required=True)
    ap.add_argument('--checkpoint-reserve-seconds', type=int, default=300)
    ap.add_argument('--status', required=True)
    ap.add_argument('--run', action='append', nargs=4, metavar=('DATASET', 'SPLIT', 'BUNDLE', 'OUT'), required=True)
    args = ap.parse_args()
    run_batch(args.run, args.model, args.model_revision, args.gpu_layers, args.limit,
              args.deadline_epoch, args.checkpoint_reserve_seconds, args.status, args.thresholds)


Writing /kaggle/working/kaggle_b0_runtime.py


In [6]:
sys.path.insert(0,'/kaggle/working')
from kaggle_b0_runtime import discover_bundles,discover_model,restore_checkpoint,load_bundle,sha256
BUNDLE_PATHS=discover_bundles(INPUT_ROOT,Path('/kaggle/working/b0_discovered_bundles'),EXPECTED_RUNS)
for key,path in BUNDLE_PATHS.items():
    manifest,_=load_bundle(path)
    print(key,manifest['count'],path)
first_manifest,_=load_bundle(next(iter(BUNDLE_PATHS.values())))
model_path=discover_model(INPUT_ROOT,first_manifest['expected_model_sha256'],MODEL_OVERRIDE)
if model_path is None:
    if not ALLOW_HUB_DOWNLOAD: raise FileNotFoundError('Add Input model Q4_K_M hoặc bật ALLOW_HUB_DOWNLOAD')
    from huggingface_hub import hf_hub_download
    model_path=Path(hf_hub_download(repo_id=first_manifest['model_repo'],filename='Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf',local_dir='/kaggle/working/qwen_gguf'))
    assert sha256(model_path)==first_manifest['expected_model_sha256'],'Downloaded model hash mismatch'
print('Model:',model_path)
if RESUME: restore_checkpoint(INPUT_ROOT,OUT_ROOT)
elif any(OUT_ROOT.rglob('records.jsonl')): raise ValueError('OUT_ROOT có records; bật RESUME hoặc chọn thư mục mới')


('devign', 'validation') 2732 /kaggle/working/b0_discovered_bundles/a53879b05dda_unpacked_b0.zip
('reveal', 'validation') 2273 /kaggle/working/b0_discovered_bundles/7ebf0e915010_unpacked_b0.zip
('devign', 'test') 2732 /kaggle/working/b0_discovered_bundles/d95a77735c13_unpacked_b0.zip
('reveal', 'test') 2274 /kaggle/working/b0_discovered_bundles/418f01de6935_unpacked_b0.zip
Model: /kaggle/input/notebooks/duchuynguyen1802/huy-qwen3coder30b-prepare-notebook/qwen_gguf/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf
No B0 checkpoint attached: start new run


In [7]:
OUT_ROOT.mkdir(parents=True,exist_ok=True)
STATUS_PATH=OUT_ROOT/'checkpoint_status.json'
cmd=[sys.executable,'-u','/kaggle/working/kaggle_b0_runtime.py','--model',str(model_path),
     '--model-revision','local-input','--gpu-layers',str(GPU_LAYERS),
     '--deadline-epoch',str(time.time()+MAX_RUNTIME_HOURS*3600),
     '--checkpoint-reserve-seconds',str(CHECKPOINT_RESERVE_SECONDS),'--status',str(STATUS_PATH)]
if LIMIT is not None: cmd+=['--limit',str(LIMIT)]
for dataset,split in EXPECTED_RUNS:
    cmd+=['--run',dataset,split,str(BUNDLE_PATHS[(dataset,split)]),str(OUT_ROOT/dataset/split)]
try:
    run_logged(cmd,'inference_batch')
finally:
    checkpoint=shutil.make_archive('/kaggle/working/grace_b0_checkpoint','zip',root_dir=OUT_ROOT)
    diagnostics_zip()
    print('Checkpoint luôn được lưu:',checkpoint)
if STATUS_PATH.exists():
    print(STATUS_PATH.read_text(encoding='utf-8'))


Bắt đầu inference_batch: ['/usr/bin/python3', '-u', '/kaggle/working/kaggle_b0_runtime.py', '--model', '/kaggle/input/notebooks/duchuynguyen1802/huy-qwen3coder30b-prepare-notebook/qwen_gguf/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M.gguf', '--model-revision', 'local-input', '--gpu-layers', '-1', '--deadline-epoch', '1793535682.54057', '--checkpoint-reserve-seconds', '300', '--status', '/kaggle/working/grace_b0_runs/checkpoint_status.json', '--run', 'devign', 'validation', '/kaggle/working/b0_discovered_bundles/a53879b05dda_unpacked_b0.zip', '/kaggle/working/grace_b0_runs/devign/validation', '--run', 'reveal', 'validation', '/kaggle/working/b0_discovered_bundles/7ebf0e915010_unpacked_b0.zip', '/kaggle/working/grace_b0_runs/reveal/validation', '--run', 'devign', 'test', '/kaggle/working/b0_discovered_bundles/d95a77735c13_unpacked_b0.zip', '/kaggle/working/grace_b0_runs/devign/test', '--run', 'reveal', 'test', '/kaggle/working/b0_discovered_bundles/418f01de6935_unpacked_b0.zip', '/kaggle/working

Tải `grace_b0_checkpoint.zip` và `grace_b0_diagnostics.zip` từ Output. Khi hết giờ, thêm checkpoint làm Input cho phiên/account tiếp theo rồi Run All; notebook tìm ở mọi đường dẫn con, đối chiếu model/prompt/label và bỏ qua ID đã hoàn thành.
`valid` là score hữu hạn; `response_valid` là token greedy thuộc 0/1. Metric raw greedy và metric score tại 0,5 được lưu riêng; AUROC/AUPRC dùng score gốc. Ngưỡng validation cho phép so xác nhận sẽ được chốt riêng, notebook không tự tối ưu bằng test.
